## Action chunking transformers

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(1), :]

In [16]:
class ACTEncoder(nn.Module):
    def __init__(self, action_dim=14, embed_dim=512, latent_dim=32, chunk_size=100):
        super().__init__()
        self.cls_token  = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.joints_proj = nn.Linear(action_dim, embed_dim)
        self.action_proj = nn.Linear(action_dim, embed_dim)
        self.pos_encoder = PositionalEncoding(embed_dim)  # max_len=500 default, not chunk_size

        encoder_layer    = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)

        self.fc_mu     = nn.Linear(embed_dim, latent_dim)
        self.fc_logvar = nn.Linear(embed_dim, latent_dim)

    def forward(self, current_joints, action_sequence):
        """
        Inputs:
            current_joints:  (batch, action_dim)
            action_sequence: (batch, chunk_size, action_dim)
        Outputs:
            mu, logvar: (batch, latent_dim)
        """
        batch_size = current_joints.size(0)

        cls_tokens     = self.cls_token.expand(batch_size, -1, -1)          # (batch, 1, 512)
        joints_emb     = self.joints_proj(current_joints).unsqueeze(1)      # (batch, 1, 512)
        actions_emb    = self.action_proj(action_sequence)                  # (batch, T, 512)
        actions_emb    = self.pos_encoder(actions_emb)                      # (batch, T, 512)

        # CLS | joints | actions  →  (batch, T+2, 512)
        sequence       = torch.cat([cls_tokens, joints_emb, actions_emb], dim=1)
        transformer_out = self.transformer(sequence)

        cls_out = transformer_out[:, 0, :]   # (batch, 512)
        return self.fc_mu(cls_out), self.fc_logvar(cls_out)


In [17]:
class ACTDecoder(nn.Module):
    def __init__(self, action_dim=14, embed_dim=512, latent_dim=32, chunk_size=100):
        super().__init__()
        self.chunk_size = chunk_size
        
        # Projections for input latents
        self.z_proj = nn.Linear(latent_dim, embed_dim)
        self.joints_proj = nn.Linear(action_dim, embed_dim)

        # Note: In reality, we will have a ResNet here processing 4 images. 
        # FOR NOW, Assume the vision backbone outputs a flattened feature map.
        self.vision_proj = nn.Linear(2048, embed_dim) 
        
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, batch_first=True)
        self.state_encoder = nn.TransformerEncoder(encoder_layer, num_layers=4)
        
        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_dim, nhead=8, batch_first=True)
        self.action_decoder = nn.TransformerDecoder(decoder_layer, num_layers=7)
        
        # The queries for the decoder (fixed position embeddings representing time)
        self.query_embed = nn.Embedding(chunk_size, embed_dim)
        
        # Final output head
        self.action_head = nn.Linear(embed_dim, action_dim)

    def forward(self, vision_features, current_joints, z):
        """
        Inputs:
            vision_features: (batch, 2048) 
            current_joints: (batch, 14)
            z: (batch, 32) -> The sampled style variable
        Outputs:
            predicted_actions: (batch, chunk_size, 14)
        """
        batch_size = z.size(0)
        
        # Project all three inputs to embed_dim (512) and add a sequence dimension (dim=1)
        # (batch, 1, 512)
        z_emb = self.z_proj(z).unsqueeze(1)
        joints_emb = self.joints_proj(current_joints).unsqueeze(1)
        vision_emb = self.vision_proj(vision_features).unsqueeze(1)
        
        # Concatenate the three embeddings along the sequence dimension to form the multimodal state
        # (batch, 3, 512)
        state_sequence = torch.cat([z_emb, joints_emb, vision_emb], dim = 1)
        
        # state_sequence through the state_encoder (This creates the 'Memory')
        # (batch, 3, 512)
        memory = self.state_encoder(state_sequence) 
        
        # Prepare the queries (representing the k timesteps we want to predict)
        queries = self.query_embed.weight.unsqueeze(0).repeat(batch_size, 1, 1) # (batch, chunk_size, 512)
        
        #  Pass queries (as target) and memory (as memory) through self.action_decoder
        # Expected shape: (batch, chunk_size, 512)
        # Q, K, V = tgt, memory, memory
        decoded_sequence = self.action_decoder(tgt=queries, memory=memory)
        
        # Project the decoded sequence to the final action dimensions using self.action_head
        # (batch, chunk_size, 14)
        predicted_actions = self.action_head(decoded_sequence) 
        
        return predicted_actions

In [18]:
class ACTModel(nn.Module):
    def __init__(self, action_dim=14, embed_dim=512, latent_dim=32, chunk_size=100):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = ACTEncoder(action_dim, embed_dim, latent_dim, chunk_size)
        self.decoder = ACTDecoder(action_dim, embed_dim, latent_dim, chunk_size)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, vision_features, current_joints, action_sequence=None):
        # Training: encoder sees the ground-truth action chunk
        if action_sequence is not None:
            mu, logvar = self.encoder(current_joints, action_sequence)
            z = self.reparameterize(mu, logvar)
        # Inference: sample from the prior N(0, I)
        else:
            mu     = torch.zeros(current_joints.size(0), self.latent_dim, device=current_joints.device)
            logvar = torch.zeros_like(mu)
            z      = mu

        predicted_actions = self.decoder(vision_features, current_joints, z)
        return predicted_actions, mu, logvar


def act_loss(predicted_actions, target_actions, mu, logvar, beta=10.0):
    """Computes CVAE loss: L1 Reconstruction + Beta * KL Divergence."""
    # L1 loss (paper-specified; avoids vanishing gradients on small errors vs MSE)
    l1_loss = F.l1_loss(predicted_actions, target_actions)
    # KL: sum over latent dim, mean over batch
    kl_div  = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
    return l1_loss + beta * kl_div


### Why L1 and not L2 loss function? 

ACT paper specifically calls out that using L2 heavily degrades the robot's performance

When errors are very small (like 0.1 cm) -> l2 -> 0.01 --> vanishing gradient issues, robot stops to be precise! So sqaured errors wont help the robot to learn precise and small movements as fast and clearly as when trained with L1 loss.

In [19]:
batch_size = 4
action_dim = 14
chunk_size = 100

dummy_vision  = torch.randn(batch_size, 2048)
dummy_joints  = torch.randn(batch_size, action_dim)
dummy_actions = torch.randn(batch_size, chunk_size, action_dim)

model = ACTModel()

print('Testing training forward pass...')
pred_actions, mu, logvar = model(dummy_vision, dummy_joints, dummy_actions)
assert pred_actions.shape == (batch_size, chunk_size, action_dim), f'Bad shape: {pred_actions.shape}'
assert mu.shape == (batch_size, model.latent_dim), f'Bad mu shape: {mu.shape}'

print('Testing loss...')
loss = act_loss(pred_actions, dummy_actions, mu, logvar)
print(f'  loss = {loss.item():.4f}')

print('Testing inference forward pass...')
inf_actions, _, _ = model(dummy_vision, dummy_joints)
assert inf_actions.shape == (batch_size, chunk_size, action_dim)
print('All checks passed.')


Testing training forward pass...
Testing loss...
  loss = 94.3884
Testing inference forward pass...
All checks passed.
